# Download EmbracePlus data for selected users

For each row of `SESSION_TABLE` this notebook downloads, from S3:
- the **raw avro** files (`raw_data/v6/*.avro`)
- the **per-minute CSVs** (`digital_biomarkers/aggregated_per_minute/*.csv`)

into `OUT_DIR/<user>/<UTC date>/<participant folder>/...`, and writes `sessions.csv` with each user's exact UTC start/end
(use those as `--start/--end` for `make_windows_16hz.py`).

Run order: cell 1 → 5. **Cell 5 starts with `DRY_RUN = True`** (only counts files and GB). Set it to `False` to really download.
Re-running is safe: files already downloaded (same size) are skipped.

In [1]:
# Cell 1 - install (run once)
%pip install boto3 pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 - configuration
import os, getpass

# Credentials: never hard-code them. Set env vars AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY, or type them when asked.
ACCESS_KEY = os.environ.get("<ACCESS_KEY>")     or getpass.getpass("AWS access key id: ")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY") or getpass.getpass("AWS secret key: ")
REGION = "us-east-1"
BUCKET = "empatica-us-east-1-prod-data"
BASE_PREFIX = "v2/2753/1/1"          # v2/<org>/<study>/<site>

# Use a folder that is NOT inside OneDrive/Dropbox (OneDrive locks files -> WinError 32)
OUT_DIR = "/mnt/e/CSE_Skills/Sem_7/BTP/Physiological-Stress-Detection/AdaStress_Integration/AdaStress_BTP/chronos-part/data_chronos_embrace"

# user, account, start_date, end_date, start_time, end_time, exclude     (dates/times are IST; None = whole day / open)
SESSION_TABLE = [
    ("user18", 2, "2026-07-31", "2026-09-15", None, None, False),
    ("user25", 5, "2026-09-15", "2026-09-23", None, None, False),
    ("user26", 5, "2026-09-24", "2026-09-30", None, None, False),
    ("user27", 5, "2026-10-01", "2026-10-05", None, None, False),

    # the five new sessions. Short recordings, so the times matter:
    # four of them share account 0 and only the window separates them.
    ("user1",  0, "2026-03-22", None, "19:05", "20:36", False),   # participant_5
    ("user1",  0, "2026-07-13", None, "16:16", "17:17", False),   # participant_15
    ("user13", 0, "2026-05-22", None, "19:00", "20:00", False),   # participant_1
    ("user14", 0, "2026-07-17", None, "20:00", "20:30", False),   # participant_13
    ("user12", 5, "2026-06-03", None, "15:40", "16:25", False),   # participant_0
]

import boto3
s3 = boto3.client("s3", aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY, region_name=REGION)
print("S3 client ready")

S3 client ready


In [3]:
# Cell 3 - turn the session table into exact UTC intervals
import pandas as pd
IST = pd.Timedelta(hours=5, minutes=30)

def session_interval(sdate, edate, stime, etime):
    """IST dates/times -> (start_utc, end_utc). None time = whole day. end_utc is exclusive."""
    edate = edate or sdate
    start = pd.Timestamp(f"{sdate} {stime or '00:00'}:00")
    if etime:
        end = pd.Timestamp(f"{edate} {etime}:00")
        if end <= start:                      # overnight, e.g. 21:00 -> 08:00
            end += pd.Timedelta(days=1)
    else:
        end = pd.Timestamp(f"{edate} 00:00:00") + pd.Timedelta(days=1)
    return start - IST, end - IST

rows = []
for user, acct, sd, ed, st, et, excl in SESSION_TABLE:
    if excl:
        continue
    s_utc, e_utc = session_interval(sd, ed, st, et)
    # S3 folders are named by UTC date -> every UTC date touched by [s_utc, e_utc)
    utc_dates = [d.strftime("%Y-%m-%d") for d in pd.date_range(s_utc.normalize(), (e_utc - pd.Timedelta(seconds=1)).normalize())]
    rows.append(dict(user=user, account=acct, start_utc=s_utc, end_utc=e_utc, utc_dates=utc_dates))
SESSIONS = pd.DataFrame(rows)

# warn when two sessions of the same account overlap in time (e.g. both whole-day on a hand-over date)
for acct, g in SESSIONS.groupby("account"):
    g = g.sort_values("start_utc").reset_index(drop=True)
    for i in range(len(g) - 1):
        if g.loc[i, "end_utc"] > g.loc[i + 1, "start_utc"]:
            print(f"WARNING account {acct}: {g.loc[i,'user']} and {g.loc[i+1,'user']} overlap "
                  f"({g.loc[i+1,'start_utc']} -> {g.loc[i,'end_utc']} UTC). Add a hand-over time (HH:MM IST) to the table.")

os.makedirs(OUT_DIR, exist_ok=True)
SESSIONS.drop(columns="utc_dates").to_csv(os.path.join(OUT_DIR, "sessions.csv"), index=False)
SESSIONS.assign(n_utc_dates=SESSIONS.utc_dates.str.len()).drop(columns="utc_dates")

,user,account,start_utc,end_utc,n_utc_dates
0,user18,2,2026-07-30 18:30:00,2026-09-15 18:30:00,48
1,user25,5,2026-09-14 18:30:00,2026-09-23 18:30:00,10
2,user26,5,2026-09-23 18:30:00,2026-09-30 18:30:00,8
3,user27,5,2026-09-30 18:30:00,2026-10-05 18:30:00,6
4,user1,0,2026-03-22 13:35:00,2026-03-22 15:06:00,1
5,user1,0,2026-07-13 10:46:00,2026-07-13 11:47:00,1
6,user13,0,2026-05-22 13:30:00,2026-05-22 14:30:00,1
7,user14,0,2026-07-17 14:30:00,2026-07-17 15:00:00,1
8,user12,5,2026-06-03 10:10:00,2026-06-03 10:55:00,1


In [4]:
# Cell 4 - S3 helpers
def s3_list(prefix, delimiter=None):
    """All keys (and common prefixes) under prefix, handling pagination."""
    keys, prefixes, token = [], [], None
    while True:
        kw = dict(Bucket=BUCKET, Prefix=prefix)
        if delimiter: kw["Delimiter"] = delimiter
        if token: kw["ContinuationToken"] = token
        r = s3.list_objects_v2(**kw)
        keys += [(o["Key"], o["Size"]) for o in r.get("Contents", [])]
        prefixes += [p["Prefix"] for p in r.get("CommonPrefixes", [])]
        if not r.get("IsTruncated"): break
        token = r["NextContinuationToken"]
    return keys, prefixes

def participant_folders(utc_date, account):
    _, pre = s3_list(f"{BASE_PREFIX}/participant_data/{utc_date}/", delimiter="/")
    names = [p.rstrip("/").split("/")[-1] for p in pre]
    return [n for n in names if n.split("-")[0].isdigit() and int(n.split("-")[0]) == account]

def download(key, size, dest, retries=5):
    import time
    if os.path.exists(dest) and os.path.getsize(dest) == size:
        return "skip"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    tmp = dest + ".part"
    for a in range(1, retries + 1):
        try:
            s3.download_file(BUCKET, key, tmp)
            os.replace(tmp, dest)
            return "ok"
        except Exception as e:
            print(f"   retry {a}/{retries} {key}: {e}")
            time.sleep(5)
    return "fail"

In [6]:
# Cell 5 - main loop
DRY_RUN = False      # True = only count files / size.  Set False to download.

SUBDIRS = {"raw_data/v6/": "raw_avro", "digital_biomarkers/aggregated_per_minute/": "per_minute"}
summary = []
for _, s in SESSIONS.iterrows():
    n = dict(ok=0, skip=0, fail=0); total = 0; n_files = 0; missing = []
    for d in s.utc_dates:
        folders = participant_folders(d, s.account)
        if not folders:
            missing.append(d); continue
        for folder in folders:
            for sub, local in SUBDIRS.items():
                keys, _ = s3_list(f"{BASE_PREFIX}/participant_data/{d}/{folder}/{sub}")
                for key, size in keys:
                    if size == 0 or key.endswith("/"): continue
                    n_files += 1; total += size
                    if not DRY_RUN:
                        dest = os.path.join(OUT_DIR, s.user, d, folder, local, os.path.basename(key))
                        n[download(key, size, dest)] += 1
    summary.append(dict(user=s.user, account=s.account, files=n_files, GB=round(total / 1e9, 2),
                        days_without_data=len(missing), **({} if DRY_RUN else n)))
    print(s.user, "done" if not DRY_RUN else "counted", "| missing UTC dates:", missing[:5], "..." if len(missing) > 5 else "")
pd.DataFrame(summary)

user18 done | missing UTC dates: ['2026-07-30', '2026-07-31', '2026-08-11', '2026-08-12', '2026-08-13'] ...
user25 done | missing UTC dates: [] 
user26 done | missing UTC dates: [] 
user27 done | missing UTC dates: [] 
user1 done | missing UTC dates: [] 
user1 done | missing UTC dates: [] 
user13 done | missing UTC dates: [] 
user14 done | missing UTC dates: [] 
user12 done | missing UTC dates: [] 


,user,account,files,GB,days_without_data,ok,skip,fail
0,user18,2,1516,0.58,29,0,1516,0
1,user25,5,497,0.27,0,0,497,0
2,user26,5,876,0.25,0,0,876,0
3,user27,5,699,0.22,0,0,699,0
4,user1,0,63,0.03,0,63,0,0
5,user1,0,61,0.04,0,61,0,0
6,user13,0,65,0.04,0,65,0,0
7,user14,0,27,0.01,0,27,0,0
8,user12,5,47,0.02,0,47,0,0


In [7]:
# Cell 6 - one ready-to-run command per user and IST day (keeps memory small)
for _, s in SESSIONS.iterrows():
    print(f"# ---- {s.user}")
    t = s.start_utc
    while t < s.end_utc:
        day_end = min((t + IST).normalize() + pd.Timedelta(days=1) - IST, s.end_utc)   # next IST midnight
        print(f'python make_windows_16hz.py --raw "{OUT_DIR}\\{s.user}" --minute "{OUT_DIR}\\{s.user}" '
              f'--out "{OUT_DIR}\\{s.user}_windows\\{(t + IST):%Y-%m-%d}" --start "{t}" --end "{day_end}"')
        t = day_end

# ---- user18
python make_windows_16hz.py --raw "/mnt/e/CSE_Skills/Sem_7/BTP/Physiological-Stress-Detection/AdaStress_Integration/AdaStress_BTP/chronos-part/data_chronos_embrace\user18" --minute "/mnt/e/CSE_Skills/Sem_7/BTP/Physiological-Stress-Detection/AdaStress_Integration/AdaStress_BTP/chronos-part/data_chronos_embrace\user18" --out "/mnt/e/CSE_Skills/Sem_7/BTP/Physiological-Stress-Detection/AdaStress_Integration/AdaStress_BTP/chronos-part/data_chronos_embrace\user18_windows\2026-07-31" --start "2026-07-30 18:30:00" --end "2026-07-31 18:30:00"
python make_windows_16hz.py --raw "/mnt/e/CSE_Skills/Sem_7/BTP/Physiological-Stress-Detection/AdaStress_Integration/AdaStress_BTP/chronos-part/data_chronos_embrace\user18" --minute "/mnt/e/CSE_Skills/Sem_7/BTP/Physiological-Stress-Detection/AdaStress_Integration/AdaStress_BTP/chronos-part/data_chronos_embrace\user18" --out "/mnt/e/CSE_Skills/Sem_7/BTP/Physiological-Stress-Detection/AdaStress_Integration/AdaStress_BTP/chronos-part/data_chronos